In [42]:
import scanpy as sc

adata = sc.read_h5ad(r"C:\Users\USER\Desktop\資訊所實習\計畫\challenge_nasal_cellxgene_230223.h5ad")

adata

AnnData object with n_obs × n_vars = 234182 × 33559
    obs: 'patient_id', 'time_point', 'covid_status', 'sex', 'cell_state', 'cell_type', 'cell_compartment', 'cell_state_woIFN', 'sequencing_library', 'Institute', 'ObjectCreateDate'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable'
    obsm: 'X_umap_rna_30pcs_6000hvgs'

In [47]:
cell_counts = adata.obs["cell_type"].value_counts()

print(cell_counts)

cell_type
Ciliated         166318
T CD8             25153
Goblet            10586
Secretory          6442
Basal              5219
Macrophage         3948
T CD4              2971
Ionocyte           2586
Club               1890
Deutorosomal       1307
NK                 1082
T MAI               962
Monocyte            868
cDC2                851
Langerhans          707
cDC Activated       590
B                   466
pDC                 443
T G/D               346
T Reg               302
Hillock             280
Squamous            279
Ductal              275
cDC1                118
AS-DC                95
Mast                 64
Melanocyte           34
Name: count, dtype: int64


In [45]:
import numpy as np

np.random.seed(123)

train_idx = []
test_idx = []

In [53]:
for ct in cell_counts.index:

    idx = np.where(
        adata.obs["cell_type"] == ct
    )[0]

    idx = np.random.permutation(idx)

    n_cells = len(idx)

    if n_cells >= 140:
        n_train = 100
        n_test = 40

    else:
        n_train = int(n_cells * 0.7)
        n_test = n_cells - n_train

    train_idx.extend(
        idx[:n_train]
    )

    test_idx.extend(
        idx[n_train:n_train+n_test]
    )

print(len(train_idx))
print(len(test_idx))

2515
1016


In [54]:
selected_idx = np.concatenate(
    [
        train_idx,
        test_idx
    ]
)

In [55]:
adata_small = adata[selected_idx].copy()

In [56]:
adata_small

AnnData object with n_obs × n_vars = 3531 × 33559
    obs: 'patient_id', 'time_point', 'covid_status', 'sex', 'cell_state', 'cell_type', 'cell_compartment', 'cell_state_woIFN', 'sequencing_library', 'Institute', 'ObjectCreateDate'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable'
    obsm: 'X_umap_rna_30pcs_6000hvgs'

In [57]:
adata_small.obs["cell_type"].value_counts()

cell_type
B                140
Basal            140
Ciliated         140
Goblet           140
Club             140
Deutorosomal     140
Ductal           140
Ionocyte         140
Hillock          140
Langerhans       140
Macrophage       140
cDC2             140
Monocyte         140
Secretory        140
NK               140
Squamous         140
T CD4            140
T G/D            140
T CD8            140
pDC              140
cDC Activated    140
T MAI            140
T Reg            140
cDC1             118
AS-DC             95
Mast              64
Melanocyte        34
Name: count, dtype: int64

In [59]:
adata_small.obs["true_label"] = (
    adata_small.obs["cell_type"].copy()
)

In [60]:
adata_small.obs["cell_type_masked"] = (
    adata_small.obs["cell_type"].copy()
)

In [61]:
adata_small.obs["is_labeled"] = False

In [62]:
adata_small.obs.iloc[
    :len(train_idx),
    adata_small.obs.columns.get_loc("is_labeled")
] = True

In [69]:
adata_small.obs["cell_type_masked"] = (
    adata_small.obs["cell_type_masked"]
    .cat.add_categories(["unknown"])
)

In [70]:
adata_small.obs.loc[
    ~adata_small.obs["is_labeled"],
    "cell_type_masked"
] = "unknown"

In [75]:
adata_small.obs[
    [
        "true_label",
        "cell_type_masked",
        "is_labeled"
    ]
]

,true_label,cell_type_masked,is_labeled
COV19_CH11283376_CTGCCTAAGTGGCACA,Ciliated,Ciliated,True
COV19_CH11283392_ATTACTCTCCTTTCTC,Ciliated,Ciliated,True
COV19_CH11283370_CCATTCGTCGAGGTAG,Ciliated,Ciliated,True
COV19_CH11931807_CCTCAGTGTTGTGGCC,Ciliated,Ciliated,True
COV19_CH11931802_GCACTCTAGTCTCCTC,Ciliated,Ciliated,True
...,...,...,...
COV19_CH11931803_GCATACACAGACAAAT,Melanocyte,unknown,False
COV19_CH11931803_ATTATCCGTTTAGCTG,Melanocyte,unknown,False
COV19_CH11931803_CTCAGAAGTAAAGGAG,Melanocyte,unknown,False
COV19_CH11931803_CTGAAACAGATAGTCA,Melanocyte,unknown,False


In [74]:
print(adata_small.obs["cell_type_masked"].value_counts())

cell_type_masked
unknown          1016
B                 100
Ciliated          100
Basal             100
Hillock           100
Deutorosomal      100
Ductal            100
Goblet            100
Langerhans        100
Ionocyte          100
Macrophage        100
Club              100
cDC Activated     100
NK                100
Squamous          100
Monocyte          100
T CD4             100
T CD8             100
T G/D             100
Secretory         100
pDC               100
cDC2              100
T Reg             100
T MAI             100
cDC1               82
AS-DC              66
Mast               44
Melanocyte         23
Name: count, dtype: int64


In [73]:
print(adata_small.obs["is_labeled"].value_counts())

is_labeled
True     2515
False    1016
Name: count, dtype: int64


In [79]:
sc.pp.normalize_total(
    adata_small,
    target_sum=1e4
)

In [80]:
sc.pp.log1p(
    adata_small
)

In [83]:
adata_small = adata_small[
    :,
    adata_small.var["vst.variable"]
].copy()

In [84]:
adata_small

AnnData object with n_obs × n_vars = 3531 × 8000
    obs: 'patient_id', 'time_point', 'covid_status', 'sex', 'cell_state', 'cell_type', 'cell_compartment', 'cell_state_woIFN', 'sequencing_library', 'Institute', 'ObjectCreateDate', 'true_label', 'cell_type_masked', 'is_labeled'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable'
    uns: 'log1p'
    obsm: 'X_umap_rna_30pcs_6000hvgs'

In [85]:
sc.pp.pca(
    adata_small,
    n_comps=50
)

In [86]:
adata_small.obsm["X_pca"].shape

(3531, 50)

In [87]:
sc.pp.neighbors(
    adata_small,
    n_neighbors=15,
    use_rep="X_pca"
)

In [89]:
adata_small.obsp["connectivities"]

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 70320 stored elements and shape (3531, 3531)>

In [ ]:
import pandas as pd

X_pca_df = pd.DataFrame(
    adata_small.obsm["X_pca"],
    index=adata_small.obs.index,
    columns=[f"PC{i+1}" for i in range(50)]
)

#X_pca_df.to_csv(
#    r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\cell_pca_features.csv"
#)

In [ ]:
label_df = adata_small.obs[
    [
        "true_label",
        "cell_type_masked",
        "is_labeled"
    ]
].copy()


#label_df.to_csv(
#    r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\cell_labels.csv"
#)

In [92]:
from scipy import sparse


adj = adata_small.obsp["connectivities"]

rows, cols = adj.nonzero()

In [ ]:
edge_df = pd.DataFrame({
    "source": rows,
    "target": cols
})

#edge_df.to_csv(
#    r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\cell_graph_edges.csv",
#    index=False
#)

In [ ]:
cell_id_df = pd.DataFrame({
    "cell_id": adata_small.obs.index,
    "node_id": range(adata_small.n_obs)
})

#cell_id_df.to_csv(
#    r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\cell_id_mapping.csv",
#    index=False
#)

In [102]:
import pandas as pd

X = pd.read_csv(r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\cell_pca_features.csv", index_col=0)

labels = pd.read_csv(r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\cell_labels.csv", index_col=0)

mapping = pd.read_csv(r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\cell_id_mapping.csv", index_col=0)

In [106]:
mapping.head()

,node_id
cell_id,
COV19_CH11283376_CTGCCTAAGTGGCACA,0
COV19_CH11283392_ATTACTCTCCTTTCTC,1
COV19_CH11283370_CCATTCGTCGAGGTAG,2
COV19_CH11931807_CCTCAGTGTTGTGGCC,3
COV19_CH11931802_GCACTCTAGTCTCCTC,4


In [107]:
print("Feature cells:", X.shape[0])
print("Label cells:", labels.shape[0])
print("Mapping cells:", mapping.shape[0])

print(
    "Feature == Label:",
    X.index.equals(labels.index)
)

print(
    "Feature == Mapping:",
    X.index.equals(mapping.index)
)

Feature cells: 3531
Label cells: 3531
Mapping cells: 3531
Feature == Label: True
Feature == Mapping: True


In [100]:
mapping.head()

,cell_id,node_id
0,COV19_CH11283376_CTGCCTAAGTGGCACA,0
1,COV19_CH11283392_ATTACTCTCCTTTCTC,1
2,COV19_CH11283370_CCATTCGTCGAGGTAG,2
3,COV19_CH11931807_CCTCAGTGTTGTGGCC,3
4,COV19_CH11931802_GCACTCTAGTCTCCTC,4
